# Experiment 003: Query2Doc + Dense Retrieval

**Date:** February 11, 2026  
**Objective:** Test Query2Doc query expansion with mDPR dense retrieval  
**Query Enhancement:** Query2Doc (LLM-based pseudo-document generation)  
**LLM:** Qwen 2.5 3B Instruct  
**Baseline:** Exp 001 (Dense, no enhancement)

**Expected Impact:**
- Improved recall for short/ambiguous queries
- Better handling of vocabulary mismatch
- Enhanced semantic context

**Reference:** Wang et al. (2023) "Query2doc: Query Expansion with Large Language Models"

## Setup

### Step 1: Clone Repository and Install Dependencies

In [ ]:
# Clone repository
!git clone https://github.com/Osmanoor/graduation.git
%cd graduation/arabic-rag-query-enhancement

# Install Java 21 (required for Pyserini)
!apt-get install -qq openjdk-21-jdk-headless

# Install Python dependencies
# Note: Using faiss-cpu (faiss-gpu has compatibility issues)
!pip install -q pyserini faiss-cpu pytrec-eval transformers torch
!pip install -q datasets  # For HuggingFace data loading
!pip install -q accelerate bitsandbytes  # For Qwen 2.5 3B

print("\n" + "="*60)
print("✓ Installation complete")
print("="*60)
print("⚠️ IMPORTANT: Restart runtime now!")
print("   1. Click 'Runtime' → 'Restart runtime'")
print("   2. Then run cells starting from 'Step 2' below")
print("="*60)

### Step 2: Mount Google Drive and Configure Environment (Run After Restart)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to project
%cd /content/graduation/arabic-rag-query-enhancement

# Configure environment
import os
import sys

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Add src to path
sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

# Verify Java
!java -version

print("\n✓ Environment configured")
print("✓ Ready to run experiment")

### Step 3: Setup Symbolic Links to Google Drive Index

In [ ]:
# Create data directory
!mkdir -p data/miracl_ar

# Link to Google Drive index
drive_base = "/content/drive/MyDrive/graduation project/colab_data"  # Update this!

!ln -sf "{drive_base}/mdpr_faiss.index" data/miracl_ar/mdpr_faiss.index
!ln -sf "{drive_base}/docid_map.pkl" data/miracl_ar/docid_map.pkl

# Verify
print("Verifying index files...")
print(f"Index exists: {os.path.exists('data/miracl_ar/mdpr_faiss.index')}")
print(f"Docid map exists: {os.path.exists('data/miracl_ar/docid_map.pkl')}")

if not os.path.exists('data/miracl_ar/mdpr_faiss.index'):
    print("\n⚠️ ERROR: Index not found!")
    print("Please update 'drive_base' path above")
else:
    print("\n✓ Index files linked successfully")

## Import Modules

In [ ]:
from src.utils.data_loader import MIRACLDataLoader
from src.retrievers.dense import mDPRRetriever
from src.enhancers.query2doc import Query2DocEnhancer
from src.evaluation.metrics import RetrievalEvaluator, save_results, save_metrics, print_metrics

import torch
from tqdm.notebook import tqdm

print("✓ Modules imported")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Load Data

In [ ]:
# Load MIRACL Arabic dev set
data_loader = MIRACLDataLoader(language="ar", split="dev")
topics, qrels = data_loader.load_all()

print(f"\nDataset Statistics:")
print(f"  Queries: {len(topics)}")
print(f"  Qrels: {len(qrels)}")

# Show sample
sample_qid = list(topics.keys())[0]
print(f"\nSample Query:")
print(f"  ID: {sample_qid}")
print(f"  Text: {topics[sample_qid]['title']}")
print(f"  Relevant docs: {len(qrels.get(sample_qid, {}))}")

## Initialize Components

In [ ]:
# Initialize Query2Doc enhancer
print("Initializing Query2Doc enhancer...")
print("This will download Qwen 2.5 3B (~6GB) on first run")

enhancer = Query2DocEnhancer(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_new_tokens=256,  # Pseudo-document length
    temperature=0.7,     # Low temperature for focused generation
    top_p=0.9
)

print("✓ Query2Doc enhancer ready")

In [ ]:
# Test enhancer on sample query
sample_query = topics[sample_qid]['title']
print(f"Testing enhancer on sample query...\n")
print(f"Original: {sample_query}")
print(f"\nGenerating pseudo-document...")

enhanced_sample = enhancer.enhance(sample_query)
print(f"\nEnhanced: {enhanced_sample[:500]}...")  # Show first 500 chars
print(f"\nLength: {len(enhanced_sample)} chars")

In [ ]:
# Initialize retriever
print("Initializing mDPR retriever...")

retriever = mDPRRetriever(
    index_path="data/miracl_ar/mdpr_faiss.index",
    docid_map_path="data/miracl_ar/docid_map.pkl",
    device="cuda"
)

print("✓ Retriever initialized")

In [ ]:
# Initialize evaluator
evaluator = RetrievalEvaluator(qrels)
print("✓ Evaluator initialized")

## Run Experiment

In [ ]:
print("="*60)
print("EXPERIMENT 003: Query2Doc + Dense Retrieval")
print("="*60)

# Prepare queries
query_ids = list(topics.keys())
query_texts = [topics[qid]['title'] for qid in query_ids]

print(f"\nTotal queries: {len(query_texts)}")

In [ ]:
# Apply Query2Doc enhancement
print(f"\nApplying Query2Doc enhancement...")
print("This will take ~15-20 minutes for all queries")
print("(~2-3 seconds per query for generation)\n")

enhanced_queries = enhancer.enhance_batch(
    query_texts, 
    query_ids,
    show_progress=True
)

print(f"\n✓ Enhanced {len(enhanced_queries)} queries")

In [ ]:
# Show enhancement examples
print("\nEnhancement Examples:\n")
for i in range(min(3, len(query_texts))):
    print(f"Query {i+1}:")
    print(f"  Original: {query_texts[i]}")
    print(f"  Enhanced: {enhanced_queries[i][:200]}...")  # First 200 chars
    print()

In [ ]:
# Run retrieval
print(f"\nRunning retrieval for {len(enhanced_queries)} queries...")
print("This will take ~5-10 minutes\n")

search_results = retriever.search(
    enhanced_queries, 
    k=100, 
    show_progress=True
)

print(f"\n✓ Retrieval complete")

In [ ]:
# Format results
print("\nFormatting results...")
results = {}

for i, qid in enumerate(tqdm(query_ids, desc="Processing")):
    results[qid] = {}
    for docid, score in search_results[i]:
        results[qid][docid] = score

print(f"✓ Formatted {len(results)} query results")

## Evaluate Results

In [ ]:
# Compute metrics
print("\nEvaluating...")
metrics = evaluator.evaluate(results)

# Print results
print_metrics(metrics, "EXPERIMENT 003: Query2Doc + Dense Results")

In [ ]:
# Compare with baseline (Exp 001)
print("\nComparison with Baseline (Exp 001 - Dense, No Enhancement):")
print("\nBaseline (Identity):")
print("  Recall@100: ~0.841")
print("  NDCG@10:    ~0.499")
print("  MRR:        ~0.xxx")

print("\nQuery2Doc:")
print(f"  Recall@100: {metrics['recall_100']:.4f}")
print(f"  NDCG@10:    {metrics['ndcg_cut_10']:.4f}")
print(f"  MRR:        {metrics['recip_rank']:.4f}")

# Calculate improvement
baseline_recall100 = 0.841
baseline_ndcg10 = 0.499

recall_improvement = ((metrics['recall_100'] - baseline_recall100) / baseline_recall100) * 100
ndcg_improvement = ((metrics['ndcg_cut_10'] - baseline_ndcg10) / baseline_ndcg10) * 100

print("\nImprovement:")
print(f"  Recall@100: {recall_improvement:+.2f}%")
print(f"  NDCG@10:    {ndcg_improvement:+.2f}%")

## Save Results

In [ ]:
import os

# Create output directory
output_dir = "results/query2doc_dense"
os.makedirs(output_dir, exist_ok=True)

# Save results in TREC format
save_results(
    results,
    f"{output_dir}/exp_003_query2doc_dense.txt",
    run_name="exp_003_query2doc"
)

# Save metrics
save_metrics(
    metrics,
    f"{output_dir}/exp_003_metrics.json"
)

print("\n✓ All results saved")

## Analysis: Query Enhancement Quality

In [ ]:
# Analyze enhancement characteristics
print("Query Enhancement Statistics:\n")

original_lengths = [len(q) for q in query_texts]
enhanced_lengths = [len(q) for q in enhanced_queries]
expansion_ratios = [e/o for o, e in zip(original_lengths, enhanced_lengths)]

import numpy as np

print(f"Original query length:")
print(f"  Mean: {np.mean(original_lengths):.1f} chars")
print(f"  Median: {np.median(original_lengths):.1f} chars")

print(f"\nEnhanced query length:")
print(f"  Mean: {np.mean(enhanced_lengths):.1f} chars")
print(f"  Median: {np.median(enhanced_lengths):.1f} chars")

print(f"\nExpansion ratio:")
print(f"  Mean: {np.mean(expansion_ratios):.2f}x")
print(f"  Median: {np.median(expansion_ratios):.2f}x")

## Summary

**Experiment:** 003 - Query2Doc + Dense Retrieval  
**Status:** Complete  

**Configuration:**
- LLM: Qwen 2.5 3B Instruct
- Max tokens: 256
- Temperature: 0.7
- Retriever: mDPR

**Results:**
- Recall@10: [Will be filled after run]
- Recall@100: [Will be filled after run]
- NDCG@10: [Will be filled after run]
- MRR: [Will be filled after run]

**Next Steps:**
1. Document results in `experiments/exp_003_query2doc_dense.md`
2. Analyze per-query improvements vs baseline
3. Test Query2Doc with BM25 (Exp 004)
4. Compare with other QE techniques